In [106]:
import json
from fastcoref import FCoref
from collections import defaultdict

In [107]:
model = FCoref(device='cuda:0')

04/11/2026 21:26:09 - INFO - 	 missing_keys: []
04/11/2026 21:26:09 - INFO - 	 unexpected_keys: []
04/11/2026 21:26:09 - INFO - 	 mismatched_keys: []
04/11/2026 21:26:09 - INFO - 	 error_msgs: []
04/11/2026 21:26:09 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


In [132]:


file = "DATA/BASIL-main/articles/2019/0d10341a-9dba-4374-a524-814c300d1611_1.json"
# file = "DATA/BASIL-main/articles/2016/7a97de89-1433-46f7-bcc2-f80b041cb9a0_2.json"
# file = "DATA/BASIL-main/articles/2016/d4a4bc34-e48b-4485-86b2-f1daf4d464dd_1.json"
def to_article_BASIL(filename: str) -> list[str]:
    ret: list[str] = []
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    #Export from nested list to a single list of sentences
    for paragraph in data["body-paragraphs"]:
        if len(paragraph) == 1:
            ret.append(paragraph[0])
        else:
            for sentence in paragraph:
                ret.append(sentence.lower()) 

    return ret
article = []
Sentence_list = to_article_BASIL(file)
#get only the first ten sentences
text = "".join(Sentence_list[:15]).lower()
text

'president trump stood firm friday on his demands for a border wall after the second white house meeting with congressional leaders this week broke up with no apparent deal, warning democrats the partial government shutdown could last "years" and saying he could even declare a "national emergency" to bypass congress if necessary.“we can call a national emergency [to build a border wall] because of the security of our country,” trump told reporters in the rose garden, during a lengthy and impromptu press conference.“i may do it,” he said, before adding, “if we can do it through a negotiated process, we’re giving it a shot.”the press conference underscored how far apart both sides are, even as trump called the meeting "productive" and suggested the standoff could end soon -- or not. he indicated he was not shifting on his demand for more than $5 billion for funding for a wall on the southern border, saying it was necessary as the border is a "dangerous, horrible disaster.""this is nation

In [133]:
preds = model.predict(texts=text)

04/11/2026 21:36:46 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 68.18 examples/s]
04/11/2026 21:36:46 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 86.58it/s]


In [168]:
cluster = preds.get_clusters()
spans: list[str] = preds.get_clusters(as_strings=False)

pronouns = {
    "i", "me", "my", "mine", "myself", "we", "us", "our", "ours", "ourselves",
    "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself",
    "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", 
    "their", "theirs", "themselves", "who", "whom", "whose", "which", "what",
    "this", "that", "these", "those", "anyone", "someone", "no one", "everyone"
}


def create_dictionary( cluster, spans):
    my_dict = defaultdict(list)
    for nouns, indexs in zip(cluster, spans):
        for noun in nouns:
            if noun not in pronouns:
                key = noun
                break
        my_dict[key].extend(indexs)
    
    return my_dict




my_dict = create_dictionary(cluster, spans)


for key in my_dict:
    print(key)




president trump
declare
a lengthy and impromptu press conference
the second white house meeting with congressional leaders this week
his demands for a border wall
a wall on the southern border
the southern border
democrats
our country
the rose garden
the partial government shutdown
a border wall
steel
the democrats'
the standoff
house speaker nancy pelosi


In [ ]:
def create_one_list(nested_list)-> list:
    return [tup for sublist in nested_list for tup in sublist]

sorted_span = sorted(create_one_list(spans), key=lambda x: x[1], reverse=True)


for span in sorted_span:
    subject = [key for key, tuples in my_dict.items() if span in tuples][0]
    
    subject = ''
    for key, tuples in my_dict.items():
        if span in tuples:
            subject = key
            break
    
    start , end = span
    text = text[:start] + subject + text[end:]    
    
text

In [ ]:
span_index: list[str] = []
for index_list in spans:
    span_index = span_index + index_list
    
span_index.sort(reverse=True)

In [ ]:

if (1479, 1499) in span_index:
    print(True)

True
